# CSE 151B: Mission 2 Testing

## Setup

In [1]:
# Install packages
!pip install transformers torch -q

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Check GPU
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU available: True
GPU name: Tesla T4


## Load Model & Data

In [2]:
# Model setup
model_name = "roneneldan/TinyStories-33M"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print(f" Model loaded on {device}")

# Training data (10 story prompts for testing)
train_stories = [
    "Write a story with: cat, forest, magic. Once upon a time, a cat wandered into a magical forest.",
    "Write a story with: boy, beach, treasure. A young boy found treasure on the beach.",
    "Write a story with: girl, library, book. A girl discovered a mysterious book in the library.",
    "Write a story with: dog, adventure, friend. A dog went on an adventure with his best friend.",
    "Write a story with: tree, summer, fun. In summer, children played under the big tree and had fun.",
    "Write a story with: moon, night, dream. Under the moon at night, she had a wonderful dream.",
    "Write a story with: river, journey, discovery. During their journey, they discovered a beautiful river.",
    "Write a story with: flower, garden, beauty. The flower in the garden showed its true beauty.",
    "Write a story with: star, sky, wish. A star shone in the sky as he made a wish.",
    "Write a story with: home, family, love. Home is where family and love are.",
]

config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  291MB            

model.safetensors: downloading bytes:           |  0.00B            

 Model loaded on cuda


## Helper Functions

In [3]:
def fine_tune(model, texts, tokenizer, device, num_epochs=3, learning_rate=1e-4, batch_size=1):
    """Fine-tune model"""
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    start_time = time.time()

    for epoch in range(num_epochs):
        total_loss = 0
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device) # Convert text to numbers
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

            optimizer.zero_grad() # Clear old gradients
            loss.backward()    # Compute new gradients
            optimizer.step()   # Update weights
            total_loss += loss.item()

        avg_loss = total_loss / len(texts)
        print(f"  Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f}")

    elapsed = time.time() - start_time
    return model, elapsed

def generate_story(model, tokenizer, device, prompt="Write a story:", max_length=50):
    """Generate a short story."""
    model.eval()
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)["input_ids"]

    with torch.no_grad():
        output_ids = model.generate(input_ids, max_length=max_length, temperature=0.7, top_p=0.9)

    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output_text

def compute_perplexity(model, texts, tokenizer, device):
    """Compute perplexity on held-out text."""
    model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss
            num_tokens = inputs["input_ids"].shape[1]
            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens if total_tokens > 0 else 0  #Compute average loss across all texts
    perplexity = torch.exp(torch.tensor(avg_loss)).item()  #Convert loss to perplexity
    return perplexity

---
# 2.1: Batch Size Impact

In [4]:
batch_sizes = [1, 5, 10]
results_batch = {}

for bs in batch_sizes:
    print(f"Batch size: {bs}")
    model_copy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    model_copy, train_time = fine_tune(model_copy, train_stories, tokenizer, device,
                                        num_epochs=2, learning_rate=1e-4, batch_size=bs) #Fine-tune on 10 stories for 2 epochs

    output = generate_story(model_copy, tokenizer, device)
    print(f"Generated: {output[:80]}...\n")

    results_batch[bs] = {"train_time": train_time}


Batch size: 1


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/2 | Loss: 2.6773


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/2 | Loss: 2.6773
  Epoch 2/2 | Loss: 0.7892
Generated: Write a story: moon, night, dream. In the moon at night, she had a wonderful dre...

Batch size: 10


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/2 | Loss: 2.6773
  Epoch 2/2 | Loss: 0.7892
Generated: Write a story: moon, night, dream. In the moon at night, she had a wonderful dre...



---
# 2.2: Learning Rate Comparison

In [5]:
learning_rates = [1e-4, 1e-5]
results_lr = {}

for lr in learning_rates:
    print(f"Learning rate: {lr}")
    model_copy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    model_copy, train_time = fine_tune(model_copy, train_stories, tokenizer, device,
                                        num_epochs=2, learning_rate=lr, batch_size=1)

    output = generate_story(model_copy, tokenizer, device)
    print(f"Generated: {output[:80]}...\n")

    results_lr[lr] = {"train_time": train_time}

Learning rate: 0.0001


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/2 | Loss: 2.6773
  Epoch 2/2 | Loss: 0.7892
Generated: Write a story: moon, night, dream. In the moon at night, she had a wonderful dre...

Learning rate: 1e-05


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/2 | Loss: 2.6168
  Epoch 2/2 | Loss: 0.9381
Generated: Write a story: tree, forest, magic. Under the tree, there was a magic tree. The ...



---
# 2.3: Layer Freezing

In [6]:
freeze_strategies = ["all", "freeze_half"]
results_freeze = {}

for strategy in freeze_strategies:
    print(f"Strategy: {strategy}")
    model_copy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    if strategy == "freeze_half":
        num_layers = len(model_copy.transformer.h) #Count layers
        for i in range(num_layers // 2): #first half
            for param in model_copy.transformer.h[i].parameters():
                param.requires_grad = False # Don't update these weights
        print(f"  Froze first {num_layers//2} layers")

    model_copy, train_time = fine_tune(model_copy, train_stories, tokenizer, device,
                                        num_epochs=2, learning_rate=1e-4, batch_size=1)

    output = generate_story(model_copy, tokenizer, device)
    print(f"Generated: {output[:80]}...\n")

    results_freeze[strategy] = {"train_time": train_time}


Strategy: all


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch 1/2 | Loss: 2.6773
  Epoch 2/2 | Loss: 0.7892
Generated: Write a story: moon, night, dream. In the moon at night, she had a wonderful dre...

Strategy: freeze_half


Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Froze first 2 layers
  Epoch 1/2 | Loss: 2.3897
  Epoch 2/2 | Loss: 0.5928
Generated: Write a story: moon, night, night, she, she, she, she. A night, she had a wonder...



---
#2.4: Early Stopping

In [7]:
#Split data
val_stories = train_stories[-2:]
train_subset = train_stories[:-2]

model_es = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model_es.train()
optimizer = torch.optim.Adam(model_es.parameters(), lr=1e-4)

best_val_loss = float('inf')
patience = 2
patience_counter = 0

print(f"Training with early stopping (patience={patience})...\n")

for epoch in range(10):
    # Training on train_subset
    train_loss = 0
    for text in train_subset:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
        outputs = model_es(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_subset)

    # Validation
    val_loss = compute_perplexity(model_es, val_stories, tokenizer, device)

    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}", end="")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        print(" Improved")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

output = generate_story(model_es, tokenizer, device)
print(f"\nGenerated: {output[:80]}...")

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training with early stopping (patience=2)...

Epoch 1 | Train Loss: 2.6913 | Val Loss: 13.8782 Improved
Epoch 2 | Train Loss: 0.7768 | Val Loss: 20.2350No improvement (1/2)
Epoch 3 | Train Loss: 0.4108 | Val Loss: 26.9797No improvement (2/2)

Early stopping at epoch 3

Generated: Write a story: girl, library, book. A girl discovered a mysterious book in the l...
